# Phase 1 — Train text-aware DTD (loss + OCR prior)

Fine-tunes the DocTamper **DTD** model with our **`CombinedTamperLoss`** (CE + Dice + boundary), and
documents how to inject the **OCR text prior** (`TextPriorFusion`) into the decoder.

## ⚠️ Read before running — this notebook is different from the OCR-recall one
- It **clones and runs the real DocTamper model**, which means it **loads the checkpoint (`.pth`) and
  `qt_table.pk` / `pks/*.pk` pickles and the ImageNet backbone `.pt` files** with `torch.load` /
  `pickle.load`. These are the *untrusted* files — run this **only in the disposable Colab session**,
  never on your laptop, and never with your Drive holding secrets.
- The upstream **full training code is not public** (the repo ships inference + a toy synthesizer). This
  notebook is a **scaffold built on the public `TamperDataset` + `seg_dtd` API** — it has **not been
  executed end-to-end by me**. Treat the data→model wiring (esp. the DCT / quant-table tensor shapes)
  as **to-be-validated**: cross-check the dataloader/forward cells against `models/eval_dtd.py`
  (lines ~43-107) which already runs correctly, and fix any shape mismatch.
- **Decoder-level OCR fusion requires a small edit to `models/dtd.py`** (you cannot inject it from
  outside the model). The last cell explains exactly where. Until that edit is done, this notebook runs
  the **loss ablation** (our loss vs the original CE) — itself a valid experiment from the proposal.

In [1]:
# --- Setup: clone DocTamper + our repo, install the DTD stack, build jpegio ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys
%cd /content
if not os.path.exists('/content/DocTamper'):
    !git clone https://github.com/qcf-568/DocTamper.git
if not os.path.exists('/content/HLCV-Project'):
    !git clone https://github.com/SamiraAbedini/HLCV-Project.git

%cd /content/DocTamper/models
!pip -q install -r requirements.txt
!pip -q install torch==1.11.0 torchvision==0.12.0 lmdb timm==0.4.12 \
    segmentation_models_pytorch==0.2.1 efficientnet_pytorch==0.7.1 easyocr
if not os.path.exists('/content/DocTamper/models/jpegio'):
    !git clone https://github.com/dwgoon/jpegio.git
    %cd jpegio
    !python setup.py install
    %cd /content/DocTamper/models

sys.path.append('/content/HLCV-Project')   # our src/ modules
print('setup done')

Mounted at /content/drive
/content
Cloning into 'DocTamper'...
remote: Enumerating objects: 363, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 363 (delta 104), reused 76 (delta 76), pack-reused 249 (from 1)
Receiving objects: 100% (363/363), 1.52 MiB | 18.08 MiB/s, done.
Resolving deltas: 100% (196/196), done.
Cloning into 'HLCV-Project'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 33 (delta 6), reused 23 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 1.97 MiB | 21.69 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/DocTamper/models
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 MB 12.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... don

In [2]:
# --- Stage data + pickles + backbones into the models/ working dir ---
# TamperDataset opens `lmdb.open(roots)` and reads `qt_table.pk` and `pks/{roots}_{minq}.pk`
# RELATIVE to the current dir, so everything must live in /content/DocTamper/models.
import os, glob, getpass
%cd /content/DocTamper/models

ROOTS = 'DocTamperV1-FCD'   # lmdb folder name == dataset key used by pks/{ROOTS}_{minq}.pk
MINQ  = 75                  # must match an existing pks/{ROOTS}_{MINQ}.pk file

# 1) qt_table.pk and pks/ ship in the repo root.
if not os.path.exists('qt_table.pk'):
    !cp ../qt_table.pk ./
if not os.path.exists('pks'):
    !cp -r ../pks ./

# 2) LMDB dataset: unzip your archive here (password typed at runtime, not stored).
ZIP_PATH = '/content/drive/MyDrive/DocTamperV1-FCD.zip'
if not glob.glob(f'{ROOTS}/**/data.mdb', recursive=True) and not os.path.exists(f'{ROOTS}/data.mdb'):
    assert os.path.exists(ZIP_PATH), f'Dataset zip not found: {ZIP_PATH}'
    pw = getpass.getpass('DocTamper zip password: ')
    !unzip -P "$pw" "{ZIP_PATH}" -d ./
    del pw
assert os.path.exists(f'{ROOTS}/data.mdb'), f'Expected {ROOTS}/data.mdb in models/ dir.'

# 3) ImageNet backbones for seg_dtd (vph_imagenet.pt, swin_imagenet.pt) come from the
#    author's checkpoints folder on Drive (same as the eval notebook). torch.load -> UNTRUSTED.
CKPT_DIR = '/content/drive/MyDrive/checkpoints'
for f in ['vph_imagenet.pt', 'swin_imagenet.pt']:
    if not os.path.exists(f) and os.path.exists(os.path.join(CKPT_DIR, f)):
        !cp "{CKPT_DIR}/{f}" ./
print('staging done. pks present:', sorted(os.listdir('pks'))[:5], '...')

/content/DocTamper/models
DocTamper zip password: ··········
Archive:  /content/drive/MyDrive/DocTamperV1-FCD.zip
   creating: ./DocTamperV1-FCD/
  inflating: ./__MACOSX/._DocTamperV1-FCD  
  inflating: ./DocTamperV1-FCD/lock.mdb  
  inflating: ./__MACOSX/DocTamperV1-FCD/._lock.mdb  
  inflating: ./DocTamperV1-FCD/data.mdb  
  inflating: ./__MACOSX/DocTamperV1-FCD/._data.mdb  
staging done. pks present: ['DocTamperV1-FCD_75.pk', 'DocTamperV1-FCD_80.pk', 'DocTamperV1-FCD_85.pk', 'DocTamperV1-FCD_90.pk', 'DocTamperV1-SCD_75.pk'] ...


In [9]:
!pip install -q lmdb

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/jpegio-0.2.4-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [12]:
!pip install -q segmentation_models_pytorch==0.2.1

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/jpegio-0.2.4-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 10.9 MB/s eta 0:00:00


In [10]:
!pip install lmdb six timm==0.4.12 segmentation_models_pytorch==0.2.1 efficientnet_pytorch==0.7.1 easyocr
%cd /content/DocTamper/models
![ -d jpegio ] || git clone https://github.com/dwgoon/jpegio.git
%cd jpegio && pip install . && %cd /content/DocTamper/models


DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/jpegio-0.2.4-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
  Using cached timm-0.4.12-py3-none-any.whl.metadata (30 kB)
  Using cached segmentation_models_pytorch-0.2.1-py3-none-any.whl.metadata (26 kB)
  Using cached efficientnet_pytorch-0.7.1.tar.gz (21 kB)
  Preparing metadata (setup.py) ... done
  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached pretrainedmodels-0.7.4.tar.gz (58 kB)
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of segmentation-models-pytorch to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install efficientnet_pytorch==0.7.1 and segmentation-models-pytorch==0.2.1 because these package versions have conflicting depe

In [14]:
# --- Build the dataset + dataloader (DocTamper's own TamperDataset) ---
# VALIDATE: confirm these match models/eval_dtd.py (the import path / class name / args).
import torch
from torch.utils.data import DataLoader
# Load TamperDataset from eval_dtd.py WITHOUT running its argparse/eval code.
import ast
src = open('/content/DocTamper/models/eval_dtd.py').read()
tree = ast.parse(src)
keep = [n for n in tree.body
        if isinstance(n, (ast.Import, ast.ImportFrom))
        or (isinstance(n, ast.ClassDef) and n.name in ('TamperDataset', 'IOUMetric'))]
ns = {}
exec(compile(ast.Module(body=keep, type_ignores=[]), 'eval_dtd_extract', 'exec'), ns)
TamperDataset = ns['TamperDataset']
print('TamperDataset loaded (script body not executed)')


train_set = TamperDataset(ROOTS, mode='train', minq=MINQ)
train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=2, drop_last=True)

batch = next(iter(train_loader))
print('batch keys:', list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f'  {k:6s} {tuple(v.shape)} {v.dtype}')
# Expected (from source): image (B,3,H,W) float | label (B,1,H,W) long {0,1}
#                         rgb=DCT (B,H,W) | q=quant table | i=quality int

TamperDataset loaded (script body not executed)


AttributeError: Caught AttributeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "eval_dtd_extract", line 98, in __getitem__
AttributeError: module 'jpegio' has no attribute 'read'


In [ ]:
# --- Model + our combined loss + optimizer ---
from dtd import seg_dtd                 # DocTamper model
from src.losses import CombinedTamperLoss
from torch.cuda.amp import autocast, GradScaler

device = 'cuda'
model = seg_dtd('', 2).to(device)

# Optional: fine-tune from the released checkpoint instead of training from scratch.
FINETUNE_PTH = 'pths/dtd_doctamper.pth'   # set to a real path, or None to skip
if FINETUNE_PTH and os.path.exists(FINETUNE_PTH):
    sd = torch.load(FINETUNE_PTH, map_location='cpu')['state_dict']   # UNTRUSTED load
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print('loaded checkpoint | missing:', len(missing), 'unexpected:', len(unexpected))

criterion = CombinedTamperLoss(lambda_dice=1.0, lambda_bound=0.5)  # ignore_index=100 (harmless on {0,1})
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scaler = GradScaler()
print('model + loss ready')

In [ ]:
# --- Training loop (loss ablation: our CombinedTamperLoss) ---
import torch.nn.functional as F

def forward_dtd(model, batch):
    """VALIDATE this against eval_dtd.py's inference call `model(data, dct, qs)`.
    If shapes mismatch, copy eval_dtd.py's exact tensor prep here."""
    data = batch['image'].to(device)
    dct  = batch['rgb'].to(device)
    qs   = batch['q'].to(device)
    return model(data, dct, qs)         # expected logits (B, 2, H, W)

EPOCHS, MAX_STEPS = 1, 200              # keep small for a first validation run
model.train()
for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        if step >= MAX_STEPS:
            break
        target = batch['label'].squeeze(1).long().to(device)   # (B, H, W)
        optimizer.zero_grad()
        with autocast():
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:           # align if head outputs H/2
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
            loss, parts = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if step % 20 == 0:
            print(f'e{epoch} s{step:04d} | total {parts["total"]:.4f} '
                  f'ce {parts["ce"]:.4f} dice {parts["dice"]:.4f} bound {parts["boundary"]:.4f}')
print('training step finished')

## Adding the OCR text prior (`TextPriorFusion`) — requires a small `dtd.py` edit

The fusion `F̂_l = φ_l([F_l, M_l])` operates on an **internal decoder feature**, so it cannot be bolted on
from outside the model. The minimal change:

1. **Pass `M_text` into `forward`.** Build the binary text mask per image with `src.text_prior` (cache it
   per LMDB index so EasyOCR runs once), add it to the batch, and extend `seg_dtd.forward(self, x, dct,
   qt, text_mask=None)` to thread it through to the decoder.
2. **Insert the module in the decoder.** In `dtd.py`'s `MID` decoder (around the `FU` module that fuses
   the visual/frequency features, per the source), add `self.fusion = TextPriorFusion(in_channels=C)`
   for that stage's channel count `C`, and call `feat = self.fusion(feat, text_mask)` before the
   remaining decoder layers.
3. **Phase-2 (TA's note):** instead of an external EasyOCR mask, add a lightweight OCR/text head on the
   Visual Perception Head features and use *its* output as `M_text` — one backbone, two heads. Select
   that head by its **recall** using `OCRCoverageMeter` (the other notebook).

Until step 2 is wired, run this notebook as the **loss ablation** (CombinedTamperLoss vs the original
CE). See [`docs/INTEGRATION.md`](../docs/INTEGRATION.md) for the full plan.

**Reminder:** never commit the dataset, `*.pth`, or `*.pk` — `.gitignore` blocks them; keep them in the
Colab session / Drive only.